# Real Data Performance

This notebook compares the performance of the EM approach to LOOCV approach using real-world datasets.

## Preview Experiment

In [ ]:
import numpy as np
import pandas as pd

from fastridge import RidgeEM, RidgeLOOCV
from experiments import run_real_data_experiments
from problems import EmpiricalDataProblem

problems = [
    EmpiricalDataProblem('abalone',    'Rings'),
    EmpiricalDataProblem('airfoil',    'scaled-sound-pressure'),
    # EmpiricalDataProblem('facebook',   'Total Interactions', nan_policy='drop_rows'),  # not git-tracked; 'like' and 'share' are also candidate targets
    EmpiricalDataProblem('concrete',   'Concrete compressive strength'),
    EmpiricalDataProblem('diabetes',   'target'),
    EmpiricalDataProblem('forest',     'area'),
    # EmpiricalDataProblem('autompg',    'mpg',   nan_policy='drop_rows'),               # not git-tracked
    # EmpiricalDataProblem('parkinsons', 'motor_UPDRS', drop=['total_UPDRS']),           # expensive
    # EmpiricalDataProblem('real_estate', 'Y house price of unit area'),                 # expensive
    EmpiricalDataProblem('student',    'G3'),
    EmpiricalDataProblem('yacht',      'Residuary_resistance'),   # much lower r2 (0.65 vs. 0.97)
    EmpiricalDataProblem('automobile', 'price', nan_policy='drop_rows'),
    # EmpiricalDataProblem('crime', 'ViolentCrimesPerPop'),                              # expensive
]

estimators = {
    'EM':     RidgeEM(),
    'CV_glm': RidgeLOOCV(alphas=100),
    'CV_fix': RidgeLOOCV(alphas=np.logspace(-10, 10, 100, endpoint=True, base=10)),
}

results = run_real_data_experiments(problems, estimators, n_iterations=10, seed=1, verbose=True)
print()

rows = []
for problem, data_result in zip(problems, results):
    em_time = data_result['EM']['time']
    cv_time = (data_result['CV_glm']['time'] + data_result['CV_fix']['time']) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: data_result[est]['r2'] for est in data_result})
    row['Speed-Up'] = cv_time / em_time
    row['p']        = data_result['EM']['p']
    row['n_train']  = data_result['EM']['n_train']
    row['n:p']      = data_result['EM']['n_train'] / data_result['EM']['p']
    rows.append(row)
pd.DataFrame(rows).sort_values('n_train', ascending=False)

## Full Experiment

In [ ]:
problems_full = [
    EmpiricalDataProblem('abalone',          'Rings'),
    EmpiricalDataProblem('airfoil',          'scaled-sound-pressure'),
    EmpiricalDataProblem('automobile',       'price',                              nan_policy='drop_rows'),
    EmpiricalDataProblem('autompg',          'mpg',                                nan_policy='drop_rows'),
    # EmpiricalDataProblem('crime',          'ViolentCrimesPerPop'),             # SVM non-converging for some run
    EmpiricalDataProblem('boston',           'medv'),
    EmpiricalDataProblem('concrete',         'Concrete compressive strength'),
    EmpiricalDataProblem('diabetes',         'target'),
    EmpiricalDataProblem('facebook',         'Total Interactions',                 nan_policy='drop_rows'),  # 'like' and 'share' are also candidate targets
    EmpiricalDataProblem('forest',           'area'),
    EmpiricalDataProblem('naval_propulsion', 'GT_compressor_decay', drop=['GT_turbine_decay']),
    EmpiricalDataProblem('naval_propulsion', 'GT_turbine_decay',    drop=['GT_compressor_decay']),
    EmpiricalDataProblem('parkinsons',       'motor_UPDRS',         drop=['total_UPDRS']),
    EmpiricalDataProblem('parkinsons',       'total_UPDRS',         drop=['motor_UPDRS']),
    EmpiricalDataProblem('real_estate',      'Y house price of unit area'),
    EmpiricalDataProblem('student',          'G3'),
    EmpiricalDataProblem('yacht',            'Residuary_resistance'),
]

estimators_full = {
    'EM':     RidgeEM(),
    'CV_glm': RidgeLOOCV(alphas=100),
    'CV_fix': RidgeLOOCV(alphas=np.logspace(-10, 10, 100, endpoint=True, base=10)),
}

results_full = run_real_data_experiments(problems_full, estimators_full,
                                         n_iterations=100, seed=1, verbose=True)
print()

rows_full = []
for problem, data_result in zip(problems_full, results_full):
    em_time = data_result['EM']['time']
    cv_time = (data_result['CV_glm']['time'] + data_result['CV_fix']['time']) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: data_result[est]['r2'] for est in data_result})
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = data_result['EM']['p']
    row['n_train']        = data_result['EM']['n_train']
    row['n:p']            = data_result['EM']['n_train'] / data_result['EM']['p']
    rows_full.append(row)
pd.DataFrame(rows_full).sort_values('n_train', ascending=False)